In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
import os

# 1. Load data from DB
engine = create_engine('sqlite:///../telco_churn.db')
df = pd.read_sql("SELECT * FROM raw_churn;", engine)

# 2. Clean up columns
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# 🚨 THE FIX: Only keep the 8 columns we actually use in the Streamlit App + the target
app_features = [
    'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 
    'InternetService', 'PaymentMethod', 'PaperlessBilling', 'TechSupport', 'Churn'
]
df_subset = df[app_features]

# 3. Encode Categorical Variables into 0s and 1s
df_subset['Churn'] = df_subset['Churn'].map({'Yes': 1, 'No': 0})
df_encoded = pd.get_dummies(df_subset, drop_first=True)

# 4. Separate Features from Target & Split Data
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Retrain the winning Logistic Regression model immediately
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_scaled, y_train)

# 7. Export the brand new, perfectly streamlined artifacts
os.makedirs('../models', exist_ok=True)
joblib.dump(model_lr, '../models/winning_churn_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X.columns.tolist(), '../models/model_features.pkl')

print("✅ Success! Your model and scaler have been streamlined to exactly match the app.")
print(f"New feature count: {len(X.columns)} columns.")

🚀 Data ready for machine learning arena!


In [2]:
# Initialize our competitive models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss')
}

# Loop through, train, and print metrics for each model
for name, model in models.items():
    print(f"=== Training {name} ===")
    
    # Train the model on the scaled training data
    model.fit(X_train_scaled, y_train)
    
    # Predict the test data
    y_pred = model.predict(X_test_scaled)
    
    # Print performance evaluation results
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

=== Training Logistic Regression ===
Accuracy: 0.8038
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407

--------------------------------------------------
=== Training Random Forest ===
Accuracy: 0.7882
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.51      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

--------------------------------------------------
=== Training XGBoost ===
Accuracy: 0.7711
              precision    recall  f1-score   support

           0       0.83      0.86      0.85  

In [3]:
# Extract the weights (coefficients) from Logistic Regression
model_lr = models["Logistic Regression"]
coefficients = model_lr.coef_[0]

# Pair them with column names
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': coefficients
}).sort_values(by='Importance', key=abs, ascending=False)

# Display the top 10 most influential features
print("--- Top 10 Features Driving Churn Decisions ---")
print(feature_importance.head(10))

--- Top 10 Features Driving Churn Decisions ---
                           Feature  Importance
1                           tenure   -1.347613
2                   MonthlyCharges   -0.851551
10     InternetService_Fiber optic    0.727745
3                     TotalCharges    0.639028
25               Contract_Two year   -0.602591
24               Contract_One year   -0.310898
21                 StreamingTV_Yes    0.249702
23             StreamingMovies_Yes    0.236368
9                MultipleLines_Yes    0.214359
28  PaymentMethod_Electronic check    0.181473


In [4]:
import joblib
import os

# Create a directory to hold saved models if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the Logistic Regression model and the scaler
joblib.dump(model_lr, '../models/winning_churn_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Save the original column names so Streamlit knows the structure
joblib.dump(X.columns.tolist(), '../models/model_features.pkl')

print("📦 Success! Model, Scaler, and Features saved inside the 'models/' folder.")

📦 Success! Model, Scaler, and Features saved inside the 'models/' folder.


In [5]:
import joblib

# Load the exact features list we exported
exported_features = joblib.load('../models/model_features.pkl')

print("--- Notebook Training Features Order ---")
for i, col in enumerate(exported_features):
    print(f"{i}: {col}")

--- Notebook Training Features Order ---
0: SeniorCitizen
1: tenure
2: MonthlyCharges
3: TotalCharges
4: gender_Male
5: Partner_Yes
6: Dependents_Yes
7: PhoneService_Yes
8: MultipleLines_No phone service
9: MultipleLines_Yes
10: InternetService_Fiber optic
11: InternetService_No
12: OnlineSecurity_No internet service
13: OnlineSecurity_Yes
14: OnlineBackup_No internet service
15: OnlineBackup_Yes
16: DeviceProtection_No internet service
17: DeviceProtection_Yes
18: TechSupport_No internet service
19: TechSupport_Yes
20: StreamingTV_No internet service
21: StreamingTV_Yes
22: StreamingMovies_No internet service
23: StreamingMovies_Yes
24: Contract_One year
25: Contract_Two year
26: PaperlessBilling_Yes
27: PaymentMethod_Credit card (automatic)
28: PaymentMethod_Electronic check
29: PaymentMethod_Mailed check
